In [1]:
import os
from datetime import datetime, timezone
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())
DB_URL = os.getenv("LOCAL_DATABASE_URL")
if DB_URL and DB_URL.startswith("postgresql://"):
    DB_URL = DB_URL.replace("postgresql://", "postgresql+psycopg2://", 1)

engine = create_engine(DB_URL)
pipeline_run_id = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")

print("Database connection berhasil")
print("Pipeline run ID:", pipeline_run_id)


Database connection berhasil
Pipeline run ID: 20260924073106


## 1. Build `customer_daily`

In [3]:
order_360 = pd.read_sql("""
    SELECT *
    FROM gold.order_360
""", engine)

order_360.to_sql(
    "_tmp_order_360",
    engine,
    if_exists="replace",
    index=False
)

customer_daily = pd.read_sql("""
WITH base AS (
    SELECT customer_id, order_date metric_date, COUNT(*) order_count, SUM(unit_quantity) unit_quantity,
           SUM(gross_merchandise_value) gross_revenue, SUM(net_revenue) net_revenue,
           SUM(refunded_amount) refund_amount, COUNT(*) FILTER (WHERE return_status <> 'NO_RETURN') return_count,
           BOOL_OR(first_order_flag) new_customer_flag, BOOL_OR(NOT first_order_flag) repeat_customer_flag
    FROM _tmp_order_360 GROUP BY customer_id, order_date
),
channels AS (
    SELECT customer_id, order_date metric_date, sales_channel,
           ROW_NUMBER() OVER (PARTITION BY customer_id, order_date ORDER BY COUNT(*) DESC, sales_channel) rn
    FROM _tmp_order_360 GROUP BY customer_id, order_date, sales_channel
),
support AS (
    SELECT "payload.customer_id" customer_id, occurred_at_utc::date metric_date,
           COUNT(DISTINCT COALESCE("payload.ticket_id", event_id)) support_contacts
    FROM silver.support_events WHERE event_type='SUPPORT_TICKET_CREATED'
    GROUP BY "payload.customer_id", occurred_at_utc::date
)
SELECT b.customer_id, b.metric_date, b.order_count, b.unit_quantity, b.gross_revenue, b.net_revenue,
       b.refund_amount, b.return_count, COALESCE(s.support_contacts,0) support_contacts, c.sales_channel active_channel,
       b.new_customer_flag, b.repeat_customer_flag
FROM base b
LEFT JOIN channels c ON c.customer_id=b.customer_id AND c.metric_date=b.metric_date AND c.rn=1
LEFT JOIN support s ON s.customer_id=b.customer_id AND s.metric_date=b.metric_date
ORDER BY b.customer_id, b.metric_date
""", engine)
print("customer_daily:", len(customer_daily), "rows /", customer_daily.customer_id.nunique(), "customers")


customer_daily: 9784 rows / 2455 customers


## 2. Contract types + pipeline_run_id

In [4]:
for c in ["order_count","unit_quantity","return_count","support_contacts"]:
    customer_daily[c] = customer_daily[c].fillna(0).astype(int)
customer_daily["pipeline_run_id"] = pipeline_run_id


## 3. Recreate target table sesuai `gold.sql`

In [5]:
gold_ddl = """CREATE SCHEMA IF NOT EXISTS gold;
DROP TABLE IF EXISTS gold.customer_daily;
CREATE TABLE gold.customer_daily (
    customer_id TEXT NOT NULL,
    metric_date DATE NOT NULL,
    order_count INTEGER NOT NULL,
    unit_quantity INTEGER NOT NULL,
    gross_revenue NUMERIC(14,2) NOT NULL,
    net_revenue NUMERIC(14,2) NOT NULL,
    refund_amount NUMERIC(14,2) NOT NULL,
    return_count INTEGER NOT NULL,
    support_contacts INTEGER NOT NULL,
    active_channel TEXT,
    new_customer_flag BOOLEAN NOT NULL,
    repeat_customer_flag BOOLEAN NOT NULL,
    pipeline_run_id TEXT NOT NULL,
    PRIMARY KEY (customer_id, metric_date)
);"""
with engine.begin() as conn:
    conn.execute(text(gold_ddl))

customer_daily.to_sql("customer_daily", engine, schema="gold", if_exists="append", index=False, method="multi")
print("gold.customer_daily berhasil dibuat.")

gold.customer_daily berhasil dibuat.


## 4. Final validation

In [6]:
df = pd.read_sql("SELECT * FROM gold.customer_daily", engine)
print("Rows:", len(df))
print("Unique customer-date:", df[["customer_id","metric_date"]].drop_duplicates().shape[0])
print("Duplicate rows:", len(df) - len(df.drop_duplicates()))
print("NULL values:", int(df.isna().sum().sum()))


Rows: 9784
Unique customer-date: 9784
Duplicate rows: 0
NULL values: 0
